# DS2002 · Public API to Clean Table

**Lab — 2026-10-09 · Fall 2026**  

---

## Lab 07 — Public API to Clean Table

Build the weather-ingestion function you will actually reuse on the midterm. It has to pull real data, survive a flaky network, say plainly when it fell back to sample data, and hand back a table that is safe to join.

The last requirement is the one people skip. A function that returns *something* no matter what is worse than one that fails loudly, because the failure ends up in your findings instead of your console.

### Part 1 — a fetch you can trust

**TODO:** finish `get_weather`. Requirements:

- sets a timeout
- retries on 429 and 5xx with an increasing delay
- raises immediately on 400 or 404 — those are your bug, not theirs
- returns a DataFrame from the `daily` block

You wrote this shape in Wednesday's studio. Rebuild it from memory if you can.

In [25]:
import requests, pandas as pd, time

BASE = 'https://archive-api.open-meteo.com/v1/archive'
RETRY_STATUSES = {429, 500, 502, 503, 504}

def get_weather(start, end, lat=38.03, lon=-78.51, tries=3):
    """Daily max temperature and total precipitation for a date range."""
    params = {
        'latitude': lat, 'longitude': lon,
        'start_date': start, 'end_date': end,
        'daily': 'temperature_2m_max,precipitation_sum',
        'timezone': 'America/New_York',
    }

    last = None
    # TODO: loop up to `tries` times
    for attempt in range(tries):
        try:
            # TODO: request with a timeout; return pd.DataFrame(r.json()['daily']) on 200
            r = requests.get(BASE, params=params, timeout=15)
            if r.status_code == 200:
                return pd.DataFrame(r.json()['daily'])
            # TODO: raise on a status that is not in RETRY_STATUSES
            if r.status_code not in RETRY_STATUSES:
                raise RuntimeError(
                    f'not retryable: {r.status_code} {r.text[:200]}')
            last = f'status {r.status_code}'
        except requests.RequestException as e:
            last = repr(e)
        # TODO: otherwise sleep (increasing each time) and try again
        time.sleep(2 ** attempt)
    # TODO: raise if every attempt failed
    raise RuntimeError(f'gave up after {tries} attempts: {last}')

### Part 2 — two home weekends

Pull weather for **2026-09-05** and **2026-10-03**. One range covering both is fine and is one request instead of two — be nice to free APIs.

If the call genuinely fails, use the fallback in Part 3, but you must record that you did.

In [26]:
# TODO: call get_weather once for a range that spans both dates, then keep just the two dates you care about

# Using the get_weather function with 9/05 and 10/03
weather = get_weather('2026-09-05', '2026-10-03', lat=38.03, lon=-78.51, tries=3)

# Using the isin function to only keep the two rows, instead of the given span
home_weekends = weather[weather['time'].isin(['2026-09-05', '2026-10-03'])]

used_fallback = False

# Calling the information
home_weekends

,time,temperature_2m_max,precipitation_sum
0,2026-09-05,29.4,2.5
28,2026-10-03,22.0,14.7


### Part 3 — the fallback, labelled as a fallback

Sample values so the notebook still runs without a network. The important part is the `source` column: whoever reads your results can tell which rows are real.

In [27]:
FALLBACK = pd.DataFrame({
    'time': ['2026-09-05', '2026-10-03'],
    'temperature_2m_max': [30.1, 18.4],
    'precipitation_sum': [0.0, 22.5],
})

# TODO: if you used the fallback, set used_fallback = True and build `wx` from FALLBACK
# TODO: add a 'source' column to `wx` -- 'open-meteo' or 'fallback sample'

# If statement - used fallback, else. Assigning wx to be the FALLBACK df or the home_weekends variable previously created
if used_fallback:
    wx = FALLBACK.copy() # Using .copy() to create indep. table for the if & else statement
    wx['source'] = 'fallback sample'
else:
    wx = home_weekends.copy()
    wx['source'] = 'open-meteo'

wx

,time,temperature_2m_max,precipitation_sum,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 4 — make it joinable

**TODO:** produce a frame with a real datetime column named `date`, plus `temp_max`, `precip_mm`, and `source`. Rename the API's column names to yours — your pipeline should not inherit somebody else's naming.

In [28]:
# TODO

# Renaming the columns - naming my pipeline (wx)
wx = wx.rename(columns={
    'time': 'date',
    'temperature_2m_max': 'temp_max',
    'precipitation_sum': 'precip_mm'
})

# Converting the date from text to a real datetime column
wx['date'] = pd.to_datetime(wx['date'])

# Calling the df
wx

,date,temp_max,precip_mm,source
0,2026-09-05,29.4,2.5,open-meteo
28,2026-10-03,22.0,14.7,open-meteo


### Part 5 — validate before you use it

This is the habit that separates a pipeline from a script. State what must be true, and find out here rather than three cells later inside a join.

**TODO:** uncomment and make them pass.

In [29]:
assert list(wx.columns) == ['date', 'temp_max', 'precip_mm', 'source']
assert len(wx) == 2, f'expected 2 rows, got {len(wx)}'
assert pd.api.types.is_datetime64_any_dtype(wx['date'])
assert wx['precip_mm'].notna().all(), 'missing precipitation'
assert (wx['precip_mm'] >= 0).all(), 'negative rainfall is not a thing'
print('weather table is safe to join.')

weather table is safe to join.


### Part 6 — answer the question

**TODO:** which of the two dates was wetter, and by how much? Print a sentence a vendor could read, not just a number.

In [30]:
# TODO
# Determining the wettest day & drywest day & difference between the two given days
wet_day = wx.loc[wx['precip_mm'].idxmax()] # indexing the max value of the precip_mm column
dry_day = wx.loc[wx['precip_mm'].idxmin()] # indexing the min value of the precip_mm column
difference = wet_day['precip_mm'] - dry_day['precip_mm'] # difference between the wetest day and the dry day in mm

print("The wettest day was", wet_day['date'].strftime("%B %d, %Y"), "with", wet_day['precip_mm'], "mm of rain.")
print("The driest day was", dry_day['date'].strftime("%B %d, %Y"), "with", dry_day['precip_mm'], "mm of rain.")
print("The difference between the two days was", difference, "mm.")

The wettest day was October 03, 2026 with 14.7 mm of rain.
The driest day was September 05, 2026 with 2.5 mm of rain.
The difference between the two days was 12.2 mm.


### Part 7 — cite it

**TODO:** print the endpoint, the coordinates, the date range, when you pulled it, and whether the data is real or fallback.

In [31]:
# TODO
from datetime import datetime

lat = 38.03
lon = -78.51

print('Source:      Open-Meteo historical archive API')
print('Endpoint:   ', BASE)
print('Coordinates:', lat, lon)
print(f"Date range:  {wx['date'].min():%Y-%m-%d} to {wx['date'].max():%Y-%m-%d}")
print('Pulled:     ', datetime.now().strftime('%Y-%m-%d %H:%M'))
print('Data type:  ', wx['source'].iloc[0])

Source:      Open-Meteo historical archive API
Endpoint:    https://archive-api.open-meteo.com/v1/archive
Coordinates: 38.03 -78.51
Date range:  2026-09-05 to 2026-10-03
Pulled:      2026-10-09 15:07
Data type:   open-meteo


### Write-up

Suppose `get_weather` returned an empty DataFrame on failure instead of raising. Walk through what happens next: you merge it onto 200 days of sales and chart rain against revenue. What does the chart show, and why would nobody catch it?

If get_weather returned an empty DataFrame instead of raising an error, the merge onto 200 days of sales would still run without an error, but the rain column would be empty for every day. The chart would look plausible, since the revenue was plotted normally. It may look like there is just a range of days it did not rain. No error would be raised, since the merge would run and the chart would render. However, this would ultimately affect the conclusion, since you are reading missing data. We would wrongly conclude that the rain does not affect the revenue.

### Stretch

Pull the same two dates for a second location — Orlando is at roughly `28.54, -81.38` — and put both cities in one long-format frame with a `city` column. That is the exact shape the midterm needs when you compare stores in different places.

In [32]:
# Stretch

# Using the get_weather function with 9/05 and 10/03
weather_cville = get_weather('2026-09-05', '2026-10-03', lat=38.03, lon=-78.51, tries=3)
weather_orlando = get_weather('2026-09-05', '2026-10-03', lat=28.54, lon=-81.38, tries=3)

# Using the isin function to only keep the two rows, instead of the given span
cville_weekends = weather[weather_cville['time'].isin(['2026-09-05', '2026-10-03'])].copy()
orlando_weekends = weather[weather_orlando['time'].isin(['2026-09-05', '2026-10-03'])].copy()

# Label each frame with its city
cville_weekends['city'] = 'Charlottesville'
orlando_weekends['city'] = 'Orlando'

# Stack them into one long-format frame - change the column names
both_cities = both_cities.rename(columns={'time': 'date',
                                          'temperature_2m_max': 'temp_max',
                                          'precipitation_sum': 'precip_mm'})

# Converting the date column from text into real dates
both_cities['date'] = pd.to_datetime(both_cities['date'])

# Printing the table
both_cities

,date,temp_max,precip_mm,city
0,2026-09-05,29.4,2.5,Charlottesville
1,2026-10-03,22.0,14.7,Charlottesville
2,2026-09-05,29.4,2.5,Orlando
3,2026-10-03,22.0,14.7,Orlando
